# Actividad 5
## Ejercicio 3

In [1]:
# Librerías a usar
import numpy as np
from scipy.linalg import expm

In [2]:
# Matriz de tasas R (4 estados)
R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
])

In [6]:
# Tasas totales de salida r_i
r_i = R.sum(axis=1)   # [5, 6, 4, 4]
r = max(r_i)          # r = 6

# Construcción de la matriz estocástica P_hat
N = R.shape[0]
P_hat = np.zeros_like(R, dtype=float)
for i in range(N):
    for j in range(N):
        if i == j:
            P_hat[i, i] = 1 - r_i[i] / r
        else:
            P_hat[i, j] = R[i, j] / r

print("Matriz P_hat:")
print(P_hat)


Matriz P_hat:
[[0.16666667 0.33333333 0.5        0.        ]
 [0.66666667 0.         0.33333333 0.        ]
 [0.         0.33333333 0.33333333 0.33333333]
 [0.16666667 0.         0.5        0.33333333]]


In [ ]:
# Función para calcular P(t) mediante uniformización
def P_t(t, P_hat, r, M=None):
    if M is None:
        rt = r * t
        M = int(max(rt + 5 * np.sqrt(rt), 20)) + 1  # +1 para asegurar
    # Inicializar suma con el término k=0
    term = np.eye(N, dtype=float)
    suma = term.copy()
    fact = 1.0
    pow_hat = np.eye(N, dtype=float)
    rt_pow = 1.0
    for k in range(1, M):
        fact *= k
        rt_pow *= r*t
        pow_hat = pow_hat @ P_hat   # P_hat^k
        term = (rt_pow / fact) * pow_hat
        suma += term
    return np.exp(-r*t) * suma

### Apartado 1

In [15]:
# Tiempos a evaluar
t_values = [0.5, 1.0, 5.0]

# Calcular y mostrar P(t)
for t in t_values:
    rt = r * t
    M = int(max(rt + 5 * np.sqrt(rt), 20)) + 1
    Pt = P_t(t, P_hat, r, M)
    print(f"P({t}) con M = {M}:")
    print(np.round(Pt, 6))
    print()

P(0.5) con M = 21:
[[0.250609 0.216965 0.386657 0.14577 ]
 [0.253135 0.238361 0.374409 0.134095]
 [0.169119 0.193615 0.420301 0.216965]
 [0.158017 0.157445 0.398332 0.286206]]

P(1.0) con M = 21:
[[0.206151 0.203902 0.39871  0.191236]
 [0.208284 0.205341 0.397899 0.188474]
 [0.196758 0.198379 0.400959 0.203902]
 [0.192046 0.193997 0.401471 0.212484]]

P(5.0) con M = 58:
[[0.199999 0.199999 0.399998 0.199999]
 [0.199999 0.199999 0.399998 0.199999]
 [0.199999 0.199999 0.399998 0.199999]
 [0.199999 0.199999 0.399998 0.199999]]



### Apartado 2

In [17]:
# Verificar Chapman-Kolmogorov: P(1) = P(0.5) * P(0.5)
P05 = P_t(0.5, P_hat, r)
P1 = P_t(1.0, P_hat, r)
P05_sq = P05 @ P05
print("P(1) calculada directamente:")
print(np.round(P1, 6))
print("\nP(0.5) * P(0.5):")
print(np.round(P05_sq, 6))
print("\nDiferencia máxima (Chapman-Kolmogorov):")
print(np.max(np.abs(P1 - P05_sq)))

P(1) calculada directamente:
[[0.206151 0.203902 0.39871  0.191236]
 [0.208284 0.205341 0.397899 0.188474]
 [0.196758 0.198379 0.400959 0.203902]
 [0.192046 0.193997 0.401471 0.212484]]

P(0.5) * P(0.5):
[[0.206151 0.203902 0.39871  0.191236]
 [0.208285 0.205341 0.3979   0.188475]
 [0.196759 0.19838  0.400959 0.203902]
 [0.192047 0.193997 0.401472 0.212485]]

Diferencia máxima (Chapman-Kolmogorov):
5.820333638384412e-07


In [16]:
# Opcional: comparar con exponencial matricial exacta (scipy) para verificar
print("\n--- Comparación con expm(R*t) (solución exacta) ---")
for t in t_values:
    P_exact = expm(R * t)
    print(f"P_exact({t}):")
    print(np.round(P_exact, 6))
    Pt_approx = P_t(t, P_hat, r)
    error = np.max(np.abs(Pt_approx - P_exact))
    print(f"Error máximo de uniformización: {error:.2e}\n")


--- Comparación con expm(R*t) (solución exacta) ---
P_exact(0.5):
[[3.185343 2.932569 3.93932  1.343365]
 [3.713492 3.777705 4.38295  1.394988]
 [1.882218 2.231305 3.561787 1.696221]
 [1.597228 1.6663   3.216014 2.166798]]
Error máximo de uniformización: 4.01e+00

P_exact(1.0):
[[30.5968   31.447888 43.75268  17.962732]
 [36.335008 37.265293 51.283559 20.715538]
 [23.694756 24.722791 35.335751 15.358095]
 [20.789622 21.765214 32.018509 14.620213]]
Error máximo de uniformización: 5.09e+01

P_exact(5.0):
[[5.35506515e+09 5.53637065e+09 7.78844323e+09 3.28068481e+09]
 [6.30958081e+09 6.52320318e+09 9.17669729e+09 3.86545172e+09]
 [4.26900193e+09 4.41353679e+09 6.20886554e+09 2.61532767e+09]
 [3.82515661e+09 3.95466428e+09 5.56333391e+09 2.34341381e+09]]
Error máximo de uniformización: 9.18e+09



## Ejercicio 4

### Ejercicio 3 con el algoritmo del ejercicio 4

In [27]:
# Matriz de tasas R (ejercicio 1)
R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

# Calcular r y P_hat
r_i = R.sum(axis=1)      # [5., 6., 4., 4.]
r = max(r_i)             # r = 6
N = R.shape[0]

P_hat = np.zeros_like(R)
for i in range(N):
    for j in range(N):
        if i == j:
            P_hat[i, i] = 1 - r_i[i] / r
        else:
            P_hat[i, j] = R[i, j] / r

print("r =", r)
print("Matriz P_hat:")
print(np.round(P_hat, 4))
print()

r = 6.0
Matriz P_hat:
[[0.1667 0.3333 0.5    0.    ]
 [0.6667 0.     0.3333 0.    ]
 [0.     0.3333 0.3333 0.3333]
 [0.1667 0.     0.5    0.3333]]



In [28]:
# Función de uniformización con tolerancia eps
def uniformization_Pt(t, P_hat, r, eps):
    rt = r * t
    # Inicialización: término k=0
    c = np.exp(-rt)
    B = c * np.eye(N)
    sum_prob = c
    A = P_hat.copy()   # A = P_hat^1
    k = 1
    # Bucle mientras cola > eps
    while sum_prob < 1 - eps:
        c = c * (rt / k)
        B = B + c * A
        A = A @ P_hat
        sum_prob += c
        k += 1
    # Al salir, el último índice usado es k-1, y la cola = 1 - sum_prob <= eps
    M = k - 1   # número máximo de términos (desde 0 hasta M)
    return B, M

In [29]:
# Tolerancia dada
eps = 1e-5
t_values = [0.5, 1.0, 5.0]

# También calculamos referencias con tolerancia muy pequeña para comparar
eps_ref = 1e-12

print("=== Resultados con tolerancia eps = 1e-5 ===\n")
for t in t_values:
    Pt, M = uniformization_Pt(t, P_hat, r, eps)
    print(f"t = {t}")
    print(f"  M = {M} (número de términos desde 0 hasta {M})")
    print("  P(t) aproximada:")
    print(np.round(Pt, 8))
    
    # Calcular referencia de alta precisión
    Pt_ref, _ = uniformization_Pt(t, P_hat, r, eps_ref)
    error = np.max(np.abs(Pt - Pt_ref))
    print(f"  Error máximo respecto a referencia (eps=1e-12): {error:.2e}\n")

=== Resultados con tolerancia eps = 1e-5 ===

t = 0.5
  M = 13 (número de términos desde 0 hasta 13)
  P(t) aproximada:
[[0.250608   0.21696392 0.38665557 0.14576911]
 [0.25313416 0.2383603  0.37440788 0.13409425]
 [0.16911882 0.19361421 0.42029966 0.21696392]
 [0.1580168  0.15744396 0.39833043 0.28620541]]
  Error máximo respecto a referencia (eps=1e-12): 1.36e-06

t = 1.0
  M = 19 (número de términos desde 0 hasta 19)
  P(t) aproximada:
[[0.20615038 0.20390128 0.39870811 0.19123506]
 [0.20828347 0.20533995 0.39789768 0.18847371]
 [0.19675775 0.19837859 0.4009572  0.20390128]
 [0.19204548 0.1939964  0.40146945 0.21248349]]
  Error máximo respecto a referencia (eps=1e-12): 2.07e-06

t = 5.0
  M = 56 (número de términos desde 0 hasta 56)
  P(t) aproximada:
[[0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999852 0.19999852 0.39999705 0.19999853]
 [0.19999852 0.19999852 0.39999705 0.19999853]]
  Error máximo respecto a referencia (eps=1e-

In [30]:
# Verificación de Chapman-Kolmogorov con la aproximación de eps=1e-5
P05, M05 = uniformization_Pt(0.5, P_hat, r, eps)
P1, M1 = uniformization_Pt(1.0, P_hat, r, eps)
P05_sq = P05 @ P05
print("=== Verificación de Chapman-Kolmogorov (con eps=1e-5) ===")
print("P(1) calculada directamente:")
print(np.round(P1, 8))
print("\nP(0.5) * P(0.5):")
print(np.round(P05_sq, 8))
print(f"\nDiferencia máxima: {np.max(np.abs(P1 - P05_sq)):.2e}")

=== Verificación de Chapman-Kolmogorov (con eps=1e-5) ===
P(1) calculada directamente:
[[0.20615038 0.20390128 0.39870811 0.19123506]
 [0.20828347 0.20533995 0.39789768 0.18847371]
 [0.19675775 0.19837859 0.4009572  0.20390128]
 [0.19204548 0.1939964  0.40146945 0.21248349]]

P(0.5) * P(0.5):
[[0.20615005 0.20390096 0.39870746 0.19123473]
 [0.20828314 0.20533963 0.39789704 0.18847339]
 [0.19675742 0.19837827 0.40095655 0.20390096]
 [0.19204515 0.19399608 0.4014688  0.21248316]]

Diferencia máxima: 6.49e-07
